# PD Disaggregation

## Why and What is PD Disaggregation?

Large Language Models (LLMs) separate Prefill (context preparation) and Decode (token generation) phases to:

Boost Efficiency: Prefill precomputes attention keys/values (KV caching) for the input sequence, enabling fast autoregressive decoding by reusing cached data—reducing compute from O(n²) to O(n) per token.
Optimize Memory: Caching KV matrices during Prefill avoids redundant recomputation, slashing memory overhead during long-sequence generation.
Leverage Hardware: Prefill exploits parallel processing for known inputs (full-sequence batched compute), while Decode optimizes latency-critical step-by-step generation.
Scale Applications: Separation allows dynamic resource allocation (e.g., high-throughput Prefill for prompts + low-latency Decode for streaming outputs), vital for real-time use cases like chatbots.

## start
### start prefill

In [ ]:

import subprocess
from rtp_llm.utils.util import wait_sever_done, stop_server
prefill_port=8090
decode_port=27001
server_process = subprocess.Popen(
        ["/opt/conda310/bin/python", "-m", "rtp_llm.start_server",
         "--checkpoint_path=/mnt/nas1/hf/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/6114e9c18dac0042fa90925f03b046734369472f/",
         "--model_type=qwen_2",
         "--role_type=PREFILL",
         f"--start_port={prefill_port}",
         "--use_local=1",
         f"--remote_rpc_server_ip=127.0.0.1:{decode_port}"
         ]
    )

wait_sever_done(server_process, prefill_port)



### start decode


In [ ]:

import subprocess
from rtp_llm.utils.util import wait_sever_done, stop_server
prefill_port=8090
decode_port=27001
server_process = subprocess.Popen(
        ["/opt/conda310/bin/python", "-m", "rtp_llm.start_server",
         "--checkpoint_path=/mnt/nas1/hf/models--Qwen--Qwen1.5-0.5B-Chat/snapshots/6114e9c18dac0042fa90925f03b046734369472f/",
         "--model_type=qwen_2",
         "--role_type=DECODE",
         f"--start_port={decode_port}",
         "--use_local=1",
         f"--remote_rpc_server_ip=127.0.0.1:{port}"
         ]
    )

wait_sever_done()

In [ ]:
import openai

prefill_port=8090
client = openai.Client(base_url=f"http://127.0.0.1:{prefill_port}/v1/chat/completions", api_key="None")

response = client.chat.completions.create(
    model="qwen/qwen2.5-0.5b-instruct",
    messages=[
        {"role": "user", "content": "List 3 countries and their capitals."},
    ],
    temperature=0,
    max_tokens=64,
)

print(f"Response: {response}")

### Advanced Configuration

Current Decode-entry P2P configuration (environment variables or matching CLI flags):

| Variable | Description | Default |
|:--------:|:-----------:|:-------:|
| **PREFILL_RETRY_TIMES** | Additional attempts for retryable Prefill multimodal preprocessing failures | `0` |
| **PREFILL_RETRY_TIMEOUT_MS** | Budget for admitting further MM retry attempts; 0 adds no separate budget, and the request deadline always applies | `0` |
| **LOAD_CACHE_TIMEOUT_MS** | KV transfer timeout in milliseconds | `60000` |
| **MAX_RPC_TIMEOUT_MS** | Default request timeout when none is supplied, in milliseconds | `7200000` |

Retries rebuild inputs from the immutable request and preserve the original start time and deadline. Decode consumes frontend-expanded token IDs; Prefill owns embedding preparation. Permanent MM errors, cancellation and request expiry stop retries. Enqueue and KV transfer are not replayed by this preprocessing retry.
